In [7]:
import numpy as np
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

In [8]:
import gym
import veins_gym

In [9]:
import os, subprocess, shutil, sys, pathlib

def source_bash_env(script_path: str):
    cmd = f"bash -lc 'source {script_path} && env -0'"
    out = subprocess.check_output(cmd, shell=True)
    for chunk in out.split(b'\x00'):
        if not chunk: continue
        k, _, v = chunk.partition(b'=')
        if k: os.environ[k.decode()] = v.decode()

omnetpp_setenv = "/home/abhay/omnetpp-5.7.1/setenv"
if os.path.exists(omnetpp_setenv):
    source_bash_env(omnetpp_setenv)
    print('Sourced:', omnetpp_setenv)
else:
    raise FileNotFoundError(f'OMNeT++ setenv not found at {omnetpp_setenv}. Update the path if installed elsewhere.')

print('opp_run path:', shutil.which('opp_run'))
try:
    ver = subprocess.run(['opp_run', '-v'], check=False, capture_output=True, text=True).stdout.splitlines()[0]
    print(ver)
except Exception as e:
    print('Warning: opp_run not callable:', e)

base = pathlib.Path.cwd()
sc_dir = (base / '..' / 'scenario').resolve()
print('scenario dir:', sc_dir)
print('omnetpp.ini exists:', (sc_dir / 'omnetpp.ini').exists())
print('run script exists:', (sc_dir / 'run').exists())
print('sumo path:', shutil.which('sumo'))

Sourced: /home/abhay/omnetpp-5.7.1/setenv
opp_run path: /home/abhay/omnetpp-5.7.1/bin/opp_run
OMNeT++ Discrete Event Simulation  (C) 1992-2021 Andras Varga, OpenSim Ltd.
scenario dir: /home/abhay/Major_Project/vanet_task_offloading/scenario
omnetpp.ini exists: True
run script exists: True
sumo path: /usr/bin/sumo


In [10]:
import os, pathlib, subprocess, shutil
base = pathlib.Path.cwd()
sc_dir = (base / '..' / 'scenario').resolve()
assert (sc_dir / 'omnetpp.ini').exists(), 'Missing scenario/omnetpp.ini'
try:
    gym.spec('veins-straight-v1')
    print('veins-straight-v1 already registered')
except Exception:
    gym.register(
        id='veins-straight-v1',
        entry_point='veins_gym:VeinsEnv',
        kwargs={
            'scenario_dir': '../scenario',
            'timeout': 7.0,
            'print_veins_stdout': False,
            'user_interface': 'Cmdenv',
            'config': 'StraightRoad',
        },
    )
    print('Registered veins-straight-v1 with scenario_dir:', sc_dir)

veins-straight-v1 already registered


In [11]:
headers = [
    'speed(m/s)', 'd0(m)', 'd1(m)', 'd2(m)', 'taskSize(MB)',
    'rsu0Busy', 'rsu1Busy', 'rsu2Busy', 'ul0(Mbps)', 'ul1(Mbps)', 'ul2(Mbps)'
]

def run_episode(agent_mode='random', verbose=False):
    env = gym.make('veins-straight-v1')
    # Fresh random seed per trial to vary OMNeT++ RNG each run
    obs = env.reset(seed=int(np.random.randint(0, 2**31 - 1)))
    # Gym API compatibility
    if isinstance(obs, tuple):
        obs = obs[0]

    done = False
    rewards = []
    steps = 0

    while not done:
        if agent_mode == 'random':
            action = env.action_space.sample()
        elif agent_mode == 'local':
            action = 0
        elif agent_mode == 'greedy':
            from nearest_free_rsu_agent import pick_action
            action = pick_action(obs)
        else:
            raise ValueError(f'Unknown agent_mode: {agent_mode}')

        prev_obs = obs
        step_out = env.step(action)
        if isinstance(step_out, tuple) and len(step_out) == 5:
            obs, reward, terminated, truncated, info = step_out
            done = bool(terminated) or bool(truncated)
        else:
            obs, reward, done, info = step_out
            done = bool(done)

        if float(obs[4]) >= 10.0:
            if verbose:
                vals = [float(x) for x in np.asarray(prev_obs).tolist()]
                print(f'Step:{steps}')
                print('  ' + ' '.join(f'{h:>11}' for h in headers))
                print('  ' + ' '.join(f'{v:>11.2f}' for v in vals))
                print(f'Action taken at step {steps}: {action}')
                print(f'Received reward: {round(float(reward), 2)}')
            rewards.append(float(reward))
        steps += 1

    env.close()
    return float(np.mean(rewards)) if rewards else 0.0

In [12]:
TRIALS = 10
AGENT_MODE = 'greedy'  # 'local' or 'greedy'

means = []
for i in range(TRIALS):
    m = run_episode(AGENT_MODE, verbose=False)
    means.append(float(m))
    print(f'Run {i+1}/{TRIALS}: mean reward = {m:.4f}')

overall = float(np.mean(means)) if means else 0.0
print('Trials:', TRIALS)
print('Agent mode:', AGENT_MODE)
print('Mean reward over runs:', overall)

/home/abhay/anaconda3/envs/mp/lib/python3.13/site-packages/gym/utils/passive_env_checker.py:181: UserWarning: WARN: The default seed argument in `Env.reset` should be `None`, otherwise the environment will by default always be deterministic. Actual default: seed='NO SEED GIVEN'
  logger.warn(
/home/abhay/anaconda3/envs/mp/lib/python3.13/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/abhay/anaconda3/envs/mp/lib/python3.13/site-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(


Run 1/10: mean reward = 0.2294
Run 2/10: mean reward = 0.2319
Run 3/10: mean reward = 0.1939
Run 4/10: mean reward = 0.2211
Run 5/10: mean reward = 0.2459
Run 6/10: mean reward = 0.2233
Run 7/10: mean reward = 0.2112
Run 8/10: mean reward = 0.2256
Run 9/10: mean reward = 0.2182
Run 10/10: mean reward = 0.2090
Trials: 10
Agent mode: greedy
Mean reward over runs: 0.22092974662794673


In [13]:
from stable_baselines3 import PPO
from train_rl_pso import RLPSOWrapper

MODEL_PATH = 'models/pso_10k.zip'  # adjust if saved elsewhere
TRIALS = 10

model = PPO.load(MODEL_PATH)

def run_episode_model(verbose=False):

    env = gym.make('veins-straight-v1')
    # Wrap with RL+PSO so model's continuous action (w,c1,c2) maps to discrete offloading
    env = RLPSOWrapper(env, num_actions=4)
    obs = env.reset(seed=int(np.random.randint(0, 2**31 - 1)))
    # Normalize reset to obs only if (obs, info) is returned
    if isinstance(obs, tuple):
        obs = obs[0]

    rewards = []
    steps = 0
    terminated = False
    truncated = False

    while not (terminated or truncated):
        prev_obs = obs
        action, _ = model.predict(obs, deterministic=True)
        step_out = env.step(action)
        if isinstance(step_out, tuple) and len(step_out) == 5:
            obs, reward, terminated, truncated, info = step_out
        else:
            obs, reward, done, info = step_out
            terminated, truncated = bool(done), False
        # Aggregate rewards only when a non-trivial task size is present
        if isinstance(obs, (list, tuple, np.ndarray)) and float(np.asarray(obs)[4]) >= 10.0:
            if verbose:
                vals = [float(x) for x in np.asarray(prev_obs).tolist()]
                print(f'Step:{steps}')
                print('  ' + ' '.join(f'{h:>11}' for h in headers))
                print('  ' + ' '.join(f'{v:>11.2f}' for v in vals))
                print(f'Action predicted: {action}')
                print(f'Received reward: {round(float(reward), 2)}')
            rewards.append(float(reward))
        steps += 1

    env.close()
    return float(np.mean(rewards)) if rewards else 0.0

# Run multiple trials and report mean
means = []
for i in range(TRIALS):
    m = run_episode_model(verbose=False)
    means.append(float(m))
    print(f'Run {i+1}/{TRIALS}: mean reward = {m:.4f}')

overall = float(np.mean(means)) if means else 0.0
print('Trials:', TRIALS)
print('Model:', MODEL_PATH)
print('Mean reward over runs:', overall)

/home/abhay/anaconda3/envs/mp/lib/python3.13/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/home/abhay/anaconda3/envs/mp/lib/python3.13/site-packages/gym/utils/passive_env_checker.py:181: UserWarning: WARN: The default seed argument in `Env.reset` should be `None`, otherwise the environment will by default always be deterministic. Actual default: seed='NO SEED GIVEN'
  logger.warn(
/home/abhay/anaconda3/envs/mp/lib/python3.13/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN:

Run 1/10: mean reward = 0.1795
Run 2/10: mean reward = 0.1999
Run 3/10: mean reward = 0.1716
Run 4/10: mean reward = 0.2111
Run 5/10: mean reward = 0.1808
Run 6/10: mean reward = 0.2396
Run 7/10: mean reward = 0.1997
Run 8/10: mean reward = 0.1894
Run 9/10: mean reward = 0.1865
Run 10/10: mean reward = 0.1842
Trials: 10
Model: models/pso_10k.zip
Mean reward over runs: 0.19422406337685386


In [14]:
# Evaluate model saved by train_rl.py (discrete RL)

MODEL_PATH = 'models/ppo_10k.zip'  # default save path in train_rl.py
TRIALS = 10

means = []
model = PPO.load(MODEL_PATH)

for i in range(TRIALS):
    
    env = gym.make('veins-straight-v1')
    obs = env.reset(seed=int(np.random.randint(0, 2**31 - 1)))
    if isinstance(obs, tuple):
        obs = obs[0]

    rewards = []
    steps = 0
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action, _ = model.predict(obs, deterministic=True)
        # If wrapper exists, it accepts continuous PSO params; else discrete actions
        step_out = env.step(int(action))
        if isinstance(step_out, tuple) and len(step_out) == 5:
            obs, reward, terminated, truncated, info = step_out
        else:
            obs, reward, done, info = step_out
            terminated, truncated = bool(done), False
        # Aggregate rewards when task size is non-trivial (>= 10 MB)
        if isinstance(obs, (list, tuple, np.ndarray)) and float(np.asarray(obs)[4]) >= 10.0:
            rewards.append(float(reward))
        steps += 1

    env.close()
    m = float(np.mean(rewards)) if rewards else 0.0
    means.append(m)
    print(f'Run {i+1}/{TRIALS}: mean reward = {m:.4f}')

overall = float(np.mean(means)) if means else 0.0
print('Trials:', TRIALS)
print('Model:', MODEL_PATH)
print('Mean reward over runs:', overall)

Run 1/10: mean reward = 0.2023
Run 2/10: mean reward = 0.2220
Run 3/10: mean reward = 0.2289
Run 4/10: mean reward = 0.2120
Run 5/10: mean reward = 0.1964
Run 6/10: mean reward = 0.2116
Run 7/10: mean reward = 0.2594
Run 8/10: mean reward = 0.1837
Run 9/10: mean reward = 0.2303
Run 10/10: mean reward = 0.2246
Trials: 10
Model: models/ppo_10k.zip
Mean reward over runs: 0.21711511157976013
